### 1. Setup, Load Data & Audit Awal
**Tujuan:** Memuat sensor_readings.csv (100.000 baris, 20 mesin) dan maintenance_logs.csv, lalu validasi tipe data kolom kunci.  
**Input:** data/raw/sensor_readings.csv, data/raw/maintenance_logs.csv (via src/config.py)  
**Output:** df_sensor (100.000×11), df_maintenance (500×8) — in-memory  
**Catatan:** Kritis: timestamp harus ter-parse sebagai datetime64[ns] — pra-syarat untuk semua operasi rolling/lag di Fase 4. machine_id harus dibaca sebagai str (bukan int) agar konsisten dengan kode downstream.  


In [ ]:
# =============================================================================
# FASE 1 — DATA INGESTION & AUDIT TIPE DATA AWAL
# =============================================================================

import sys
import numpy as np
import pandas as pd
from pathlib import Path

# --- [1] SETUP sys.path agar src/config.py dapat di-import ---
# Notebook berada di: notebooks/fase_1_ingestion/
# ROOT ML = dua level ke atas dari direktori ini
NOTEBOOK_DIR = Path().resolve()
ML_ROOT      = NOTEBOOK_DIR.parent.parent          # → machine_learning/
SRC_PATH     = ML_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

# --- [2] Import config (single source of truth) ---
from config import SENSOR_FILE, MAINTENANCE_FILE

print(f"✅  sys.path diperluas: {SRC_PATH}")
print(f"📂  SENSOR_FILE      : {SENSOR_FILE}")
print(f"📂  MAINTENANCE_FILE : {MAINTENANCE_FILE}")

# =============================================================================
# [3] LOAD DATA
# =============================================================================

# -- Sensor readings --
df_sensor = pd.read_csv(
    SENSOR_FILE,
    parse_dates=["timestamp"],
    dtype={"machine_id": str}          # Pastikan machine_id terbaca sebagai string
)

# -- Maintenance logs --
df_maintenance = pd.read_csv(
    MAINTENANCE_FILE,
    parse_dates=["date"]
)

# =============================================================================
# [4] AUDIT REPORT
# =============================================================================

SEP = "=" * 60
sep = "-" * 60

print(f"\n{SEP}")
print("  📊  DATA INGESTION — AUDIT REPORT")
print(SEP)

# --- Shape ---
print(f"\n{sep}")
print("  SHAPE")
print(sep)
print(f"  df_sensor      : {df_sensor.shape}   (rows × cols)")
print(f"  df_maintenance : {df_maintenance.shape}   (rows × cols)")

# --- dtypes df_sensor ---
print(f"\n{sep}")
print("  df_sensor — DTYPES")
print(sep)
print(df_sensor.dtypes.to_string())

# --- dtypes df_maintenance ---
print(f"\n{sep}")
print("  df_maintenance — DTYPES")
print(sep)
print(df_maintenance.dtypes.to_string())

# --- Konfirmasi kolom datetime ---
is_timestamp_dt = pd.api.types.is_datetime64_any_dtype(df_sensor["timestamp"])
is_date_dt      = pd.api.types.is_datetime64_any_dtype(df_maintenance["date"])

print(f"\n{sep}")
print("  KONFIRMASI TIPE KOLOM KUNCI")
print(sep)
print(f"  df_sensor['timestamp']     is datetime64 → {is_timestamp_dt}")
print(f"  df_maintenance['date']     is datetime64 → {is_date_dt}")

# --- Konfirmasi machine_id sebagai string/object ---
is_machine_id_str = df_sensor["machine_id"].dtype == object
print(f"  df_sensor['machine_id']    is string/object → {is_machine_id_str}")

print(f"\n{SEP}")
print("  ✅  Ingestion selesai. Memuat Data & Audit Tipe Data Awal")
print(SEP)

---
## 🔍 Audit Lanjutan — Missing Values · Distribusi Failure · Statistik Sensor
Cell ini menjalankan tiga pemeriksaan mendalam terhadap data yang sudah dimuat di cell sebelumnya.

| Bagian | Fokus |
|--------|-------|
| **1** | Missing values per kolom (jumlah & %) |
| **2** | Distribusi label `failure` + statistik per mesin |
| **3** | Range & statistik deskriptif kolom sensor numerik |

### 2. Audit Lanjutan — Missing Values · Distribusi Failure · Statistik Sensor
**Tujuan:** Menjalankan tiga pemeriksaan mendalam: missing values per kolom, distribusi label failure, dan statistik deskriptif sensor numerik.  
**Input:** df_sensor, df_maintenance (dari Cell 1)  
**Output:** Laporan audit ke stdout (missing values, distribusi label, statistik sensor)  
**Catatan:** Distribusi label: failure=0 (99.944%) vs failure=1 (0.056%) — imbalance ekstrem yang menjadi motivasi utama SSBS di Fase 6. Nilai negatif pada vibration (min=-0.09) adalah Defect DFT-02 yang diselesaikan di Fase 5 dengan clip ke 0.  

🎯 **Dipakai di:** Paper IEEE Section III.A (Dataset Description)

In [ ]:
# =============================================================================
# FASE 1 — AUDIT LANJUTAN
# Bagian 1 · Missing Values
# Bagian 2 · Distribusi Label Failure
# Bagian 3 · Range & Statistik Sensor
# =============================================================================

SEP = "=" * 65
sep = "-" * 65

print(SEP)
print("  🔍  AUDIT LANJUTAN")
print(SEP)

# ─────────────────────────────────────────────────────────────────
# BAGIAN 1 — MISSING VALUES
# ─────────────────────────────────────────────────────────────────
print(f"\n{'▌ BAGIAN 1':^65}")
print("  MISSING VALUES")
print(sep)

def _missing_report(df, name):
    """Hitung missing values; tampilkan hanya kolom dengan NaN > 0."""
    total    = len(df)
    mv_count = df.isnull().sum()
    mv_pct   = (mv_count / total * 100).round(4)
    report   = (
        pd.DataFrame({"missing_count": mv_count, "missing_pct_%": mv_pct})
        .query("missing_count > 0")
        .sort_values("missing_count", ascending=False)
    )
    print(f"\n  [{name}]  total rows = {total:,}")
    if report.empty:
        print("  ✅  Tidak ada missing values — semua kolom lengkap.")
    else:
        print(report.to_string())

_missing_report(df_sensor,      "df_sensor")
_missing_report(df_maintenance, "df_maintenance")

# ─────────────────────────────────────────────────────────────────
# BAGIAN 2 — DISTRIBUSI LABEL FAILURE
# ─────────────────────────────────────────────────────────────────
print(f"\n{sep}")
print(f"{'▌ BAGIAN 2':^65}")
print("  DISTRIBUSI LABEL FAILURE (df_sensor)")
print(sep)

# --- Value counts & persentase ---
failure_counts = df_sensor["failure"].value_counts().sort_index()
failure_pct    = (
    df_sensor["failure"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(4)
)
failure_summary = pd.DataFrame({
    "label" : failure_counts.index,
    "count" : failure_counts.values,
    "pct_%" : failure_pct.values,
})
print(f"\n{failure_summary.to_string(index=False)}")

# --- Statistik per mesin ---
df_failed             = df_sensor[df_sensor["failure"] == 1]
n_machines_total      = df_sensor["machine_id"].nunique()
n_machines_failed     = df_failed["machine_id"].nunique()
avg_failure_per_machine = (
    df_failed
    .groupby("machine_id")["failure"]
    .count()
    .mean()
)

print(f"\n  Jumlah mesin unik (total)            : {n_machines_total}")
print(f"  Mesin yang pernah failure=1          : {n_machines_failed}")
print(f"  Rata-rata kejadian failure per mesin : {avg_failure_per_machine:.2f} kali")

# ─────────────────────────────────────────────────────────────────
# BAGIAN 3 — RANGE & STATISTIK SENSOR
# ─────────────────────────────────────────────────────────────────
print(f"\n{sep}")
print(f"{'▌ BAGIAN 3':^65}")
print("  STATISTIK DESKRIPTIF KOLOM SENSOR NUMERIK")
print(sep)

SENSOR_COLS = [
    "temperature", "vibration", "pressure", "rpm",
    "power_consumption", "noise_level", "humidity", "operating_hours",
]
PHYSICALLY_POSITIVE_COLS = ["temperature", "pressure", "rpm"]

# Ambil hanya kolom yang benar-benar ada (defensive)
sensor_cols_present = [c for c in SENSOR_COLS if c in df_sensor.columns]

print(f"\n  Kolom yang diperiksa: {sensor_cols_present}")
print()
print(df_sensor[sensor_cols_present].describe().round(4).to_string())

# --- Pengecekan nilai fisik tidak valid (<= 0) ---
print(f"\n{sep}")
print("  CEK NILAI FISIK TIDAK VALID (<= 0)")
print(f"  Kolom: {PHYSICALLY_POSITIVE_COLS}")
print(sep)

any_invalid = False
for col in PHYSICALLY_POSITIVE_COLS:
    if col not in df_sensor.columns:
        print(f"  ⚠️   Kolom '{col}' tidak ditemukan di df_sensor — skip.")
        continue
    n_invalid = int((df_sensor[col] <= 0).sum())
    flag = "⚠️  DITEMUKAN" if n_invalid > 0 else "✅  OK"
    print(f"  {flag}  '{col}': {n_invalid:,} baris dengan nilai <= 0")
    if n_invalid > 0:
        any_invalid = True

print(f"\n{SEP}")
if any_invalid:
    print("  ⚠️   Ada nilai fisik tidak valid — perlu penanganan di Fase Preprocessing.")
else:
    print("  ✅  Semua nilai fisik dalam rentang valid.")
print(SEP)

---
## 🕒 Audit Temporal Integrity
Memverifikasi konsistensi dimensi waktu pada `df_sensor` sebelum data digunakan untuk feature engineering berbasis time-series.

| Bagian | Fokus |
|--------|-------|
| **1** | Rentang waktu global (min, max, durasi total) |
| **2** | Duplikat kombinasi `(timestamp, machine_id)` |
| **3** | Distribusi jumlah baris per mesin |
| **4** | Gap temporal > 1 jam per mesin |

### 3. Audit Temporal Integrity
**Tujuan:** Memverifikasi konsistensi dimensi waktu: rentang global, duplikat timestamp, distribusi baris per mesin, dan gap temporal >1 jam.  
**Input:** df_sensor (dari Cell 1)  
**Output:** Laporan temporal integrity ke stdout  
**Catatan:** Semua pemeriksaan PASSED: 0 duplikat timestamp, 5.000 baris per mesin (balanced), 0 gap >1 jam. Rentang waktu: 1 Jul 2025 – 25 Jan 2026 (208 hari).  


In [ ]:
# =============================================================================
# FASE 1 — AUDIT TEMPORAL INTEGRITY
# Bagian 1 · Rentang Waktu Global
# Bagian 2 · Duplikat Timestamp per Mesin
# Bagian 3 · Distribusi Baris per Mesin
# Bagian 4 · Gap Temporal per Mesin (> 1 jam)
# =============================================================================

SEP = "=" * 65
sep = "-" * 65

GAP_THRESHOLD = pd.Timedelta(hours=1)   # threshold gap: dapat diubah tanpa menyentuh logika

print(SEP)
print("  🕒  AUDIT TEMPORAL INTEGRITY — df_sensor")
print(SEP)

# ─────────────────────────────────────────────────────────────────
# BAGIAN 1 — RENTANG WAKTU GLOBAL
# ─────────────────────────────────────────────────────────────────
print(f"\n{'▌ BAGIAN 1':^65}")
print("  RENTANG WAKTU GLOBAL")
print(sep)

ts_min      = df_sensor["timestamp"].min()
ts_max      = df_sensor["timestamp"].max()
ts_duration = ts_max - ts_min

print(f"\n  Timestamp minimum  : {ts_min}")
print(f"  Timestamp maksimum : {ts_max}")
print(f"  Total durasi       : {ts_duration.days} hari "
      f"({ts_duration.days / 30.44:.1f} bulan approx.)")

# ─────────────────────────────────────────────────────────────────
# BAGIAN 2 — DUPLIKAT TIMESTAMP PER MESIN
# ─────────────────────────────────────────────────────────────────
print(f"\n{sep}")
print(f"{'▌ BAGIAN 2':^65}")
print("  DUPLIKAT (timestamp, machine_id)")
print(sep)

dup_mask  = df_sensor.duplicated(subset=["timestamp", "machine_id"], keep=False)
n_dup     = int(dup_mask.sum())
df_dup    = df_sensor[dup_mask]

print(f"\n  Jumlah baris duplikat ditemukan: {n_dup:,}")

if n_dup == 0:
    print("  ✅  Tidak ada duplikat — setiap (timestamp, machine_id) unik.")
else:
    print(f"  ⚠️   Terdapat {n_dup:,} baris duplikat — perlu deduplikasi.")
    print(f"\n  Contoh 5 baris duplikat pertama:")
    print(
        df_dup[["timestamp", "machine_id"]]
        .sort_values(["machine_id", "timestamp"])
        .head(5)
        .to_string(index=True)
    )

# ─────────────────────────────────────────────────────────────────
# BAGIAN 3 — DISTRIBUSI BARIS PER MESIN
# ─────────────────────────────────────────────────────────────────
print(f"\n{sep}")
print(f"{'▌ BAGIAN 3':^65}")
print("  DISTRIBUSI BARIS PER MESIN")
print(sep)

rows_per_machine = df_sensor.groupby("machine_id").size()
dist_summary     = rows_per_machine.value_counts().sort_index()

print(f"\n  Jumlah mesin unik: {len(rows_per_machine)}")
print(f"\n  Distribusi jumlah baris (baris_count → jumlah_mesin):")

dist_df = dist_summary.rename_axis("baris_per_mesin").reset_index(name="jumlah_mesin")
print(dist_df.to_string(index=False))

min_rows = int(rows_per_machine.min())
max_rows = int(rows_per_machine.max())
selisih  = max_rows - min_rows

print(f"\n  Min baris per mesin : {min_rows:,}")
print(f"  Max baris per mesin : {max_rows:,}")
print(f"  Selisih (max - min) : {selisih:,}")

if selisih == 0:
    print("  ✅  Semua mesin memiliki jumlah baris yang sama (balanced).")
else:
    print("  ⚠️   Terdapat ketidakseimbangan jumlah baris antar mesin.")

# ─────────────────────────────────────────────────────────────────
# BAGIAN 4 — GAP TEMPORAL PER MESIN (> 1 JAM)
# ─────────────────────────────────────────────────────────────────
print(f"\n{sep}")
print(f"{'▌ BAGIAN 4':^65}")
print(f"  GAP TEMPORAL PER MESIN  (threshold: {GAP_THRESHOLD})")
print(sep)

# Urutkan satu kali, hitung diff per group
df_sorted = df_sensor.sort_values(["machine_id", "timestamp"])

gap_series = (
    df_sorted
    .groupby("machine_id")["timestamp"]
    .diff()                              # selisih antar baris berurutan per mesin
)

# Mask baris yang gap-nya melebihi threshold
gap_mask       = gap_series > GAP_THRESHOLD
df_with_gaps   = df_sorted[gap_mask].copy()
df_with_gaps["gap_duration"] = gap_series[gap_mask]

if df_with_gaps.empty:
    print(f"\n  ✅  Tidak ada gap > {GAP_THRESHOLD} — deret waktu kontinyu untuk semua mesin.")
else:
    gap_per_machine = (
        df_with_gaps
        .groupby("machine_id")["gap_duration"]
        .agg(
            total_gap_count="count",
            max_gap=lambda x: str(x.max()),
        )
        .sort_values("total_gap_count", ascending=False)
    )
    n_machines_with_gaps = len(gap_per_machine)
    total_gaps           = int(df_with_gaps.shape[0])

    print(f"\n  ⚠️   Total gap ditemukan  : {total_gaps:,} baris")
    print(f"  ⚠️   Mesin dengan gap     : {n_machines_with_gaps} mesin")
    print(f"\n  Detail per mesin (hanya mesin yang ada gap-nya):")
    print(gap_per_machine.to_string())

print(f"\n{SEP}")
print("  ✅  Audit Temporal Integrity selesai.")
print(SEP)